In [5]:
import pandas as pd
import numpy as np
columns = ["Sex","Length","Diameter","Height","Whole_weight",
           "Shucked_weight","Viscera_weight","Shell_weight","Rings"]
df = pd.read_csv("abalone.data", names=columns)
print(f"Number of rows:{len(df)}")
print(f"Columns:{df.columns.tolist()}")
print(df.head())
df['y'] = df['Rings']+1.5
X_raw = df[["Shell_weight","Diameter","Whole_weight"]].values
y = df['y'].values.reshape(-1,1)

Number of rows:4177
Columns:['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


In [ ]:
#chosen shell weight,diameter and whole_weight as features because they are storng predictors
#shell weight-shell thickens and hardens significantly as the abalone ages so strong predictor
#diamater-good metric for growth size
#whole_weight-represents full class

In [6]:
split_idx = int(0.8 * len(df))
X_train_raw, X_test_raw = X_raw[:split_idx], X_raw[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [7]:
mean = np.mean(X_train_raw, axis=0)
std = np.std(X_train_raw, axis=0)

In [10]:
X_train =(X_train_raw-mean)/std
X_test =(X_test_raw-mean)/std

In [ ]:
#normalisation is needed because different features have units whose values can vary significantly
#because of different scales,we need uniformity toi get them in a same range hence normalise

In [11]:
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (3341, 3), Test shape: (836, 3)


In [12]:
def forward(X, w, b):
    return X@w+b
def mse(y, y_hat):
    return np.mean((y-y_hat)**2)

In [ ]:
#square to help smaller errors and to make errors positive
#large errors become more expensive because square of a large value is even larger

In [13]:
def grad_w(X, y, y_hat):
    N = len(y)
    return (2/N)*X.T@(y_hat-y)

def grad_b(y, y_hat):
    return np.mean(2*(y_hat-y))

In [ ]:
#gradient is the slope including direction
#gradient is uphill pointing so reducing makes downhill thus decrease

In [ ]:
#large gradient means steep slope
#if learning rate is too marge,we may miss downhill points and move back and forth

In [14]:
np.random.seed(42)
w = np.random.randn(3, 1) * 0.01
b = 0.0
lr = 0.01
epochs = 1000

for epoch in range(epochs):
    y_hat = forward(X_train,w,b)
    loss = mse(y_train,y_hat)
    dw = grad_w(X_train,y_train, y_hat)
    db = grad_b(y_train,y_hat)
    w -= lr * dw
    b -= lr * db
    if epoch % 100 == 0:
        print(f"Epoch {epoch},Loss:{loss:.4f}")

Epoch 0,Loss:144.2610
Epoch 100,Loss:9.2494
Epoch 200,Loss:6.7935
Epoch 300,Loss:6.6236
Epoch 400,Loss:6.5145
Epoch 500,Loss:6.4236
Epoch 600,Loss:6.3471
Epoch 700,Loss:6.2828
Epoch 800,Loss:6.2286
Epoch 900,Loss:6.1829


In [15]:
test_preds = forward(X_test, w, b)
test_mse = mse(y_test, test_preds)
test_mae = np.mean(np.abs(y_test - test_preds))
print(f"\nTest MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")
for i in range(5):
    print(f"True Age: {y_test[i][0]:.1f}|Predicted:{test_preds[i][0]:.1f}")


Test MSE: 4.8203
Test MAE: 1.6706
True Age: 13.5|Predicted:11.1
True Age: 15.5|Predicted:9.9
True Age: 14.5|Predicted:10.5
True Age: 14.5|Predicted:11.7
True Age: 13.5|Predicted:11.9


In [ ]:
#For actual age of 15.5,the model predicted 9.9,and for 14.5,it predicted 10.5.This
#seems systematically wrong

#Tendency toward the Mean:The model's predictions are much closer to each other than the true ages (ranging from 13.5 to 15.5).
#The Role of Bias: The bias term acts as a starting salary/age. Because the model is linear and the weights are optimized to reduce the overall Mean Squared Error (MSE),
# it often "biases" its predictions toward the average of the training set to minimize the risk of massive penalties.